# Extending **MesoMath**

## The "Vertical Problem": Defining Height

In many Mesopotamian mathematical problems, vertical measurements (height or depth) are treated with 
specific units that, while sharing the same names as lengths, behave differently in calculations.

Let's define a custom class `bh` (Babylonian Height) that inherits from the length system 
(`bl` or `Blen`) but fixes the base unit to the *kuš3* (cubit).

In [1]:
from mesomath.npvs import Blen, Bsur, Bvol, Bcap
from mesomath.nb_utils import setup_scribal_environment

# Initialize cuneiform support
setup_scribal_environment()

In [2]:
# We define our custom height class
class bh(Blen):
    title: str = "Babylonian Height Measurement"
    ubase: int = 1  # Fixed to 'kus' (cubit)

## Seamless Interaction

Thanks to the polymorphic design of **MesoMath**, your custom classes are recognized by the core engine. 
    You can multiply a standard surface (`Bsur`) by your new height (`bh`) to obtain a volume (`Bvol`) 
without any type errors.

In [3]:
# 1. Define a surface of 1 sar
area = Bsur('1 sar')

# 2. Define a height using our custom class
height = bh('1 kus')

# 3. Calculate volume (Area * Height)
# The system recognizes 'bh' as a valid length for this operation.
volume = area * height

print(f"Volume: {volume}") 
# Output: 1 sar

Volume: 1 sar


## Automatic Utilities

By inheriting from `MesoM` (via `Blen`), your new class automatically gains all the new administrative and diagnostic tools of this version:

1. **Metrological Lists/Tables**: Generate tables for your custom units instantly.
   

In [4]:
bh.metrolist('1 kus', '5 kus', '1 kus', verbose=True, ubase=None)


Babylonian Height Measurement
danna <-30- us <-60- ninda <-12- kus <-30- susi
|Measurement         | Sexag. (kus)    | Reciprocal  |
|--------------------|-----------------|-------------|
|1 kus               | 1               | 1           |
|2 kus               | 2               | 30          |
|3 kus               | 3               | 20          |
|4 kus               | 4               | 15          |
|5 kus               | 5               | 12          |


In [5]:
bh.metrolist('10 susi', '2 kus', '5 susi', verbose=True, width=30,fractions=2,actual=True)


Babylonian Height Measurement
danna <-30- UŠ <-60- ninda <-12- kuš3 <-30- šu-si
|Measurement                   | Sexag. (kuš3)   | Reciprocal  |
|------------------------------|-----------------|-------------|
|1/3 kuš3                      | 20              | 3           |
|1/2 kuš3                      | 30              | 2           |
|2/3 kuš3                      | 40              | 1:30        |
|5/6 kuš3                      | 50              | 1:12        |
|1 kuš3                        | 1               | 1           |
|1 1/6 kuš3                    | 1:10            | --igi nu--  |
|1 1/3 kuš3                    | 1:20            | 45          |
|1 1/2 kuš3                    | 1:30            | 40          |
|1 2/3 kuš3                    | 1:40            | 36          |
|1 5/6 kuš3                    | 1:50            | --igi nu--  |
|1/6 ninda                     | 2               | 30          |


In [6]:
           
bh.prtsex = 1
bh.metrolist('10 susi', '2 kus', '5 susi', verbose=True, width=30,fractions=2,actual=True)


Babylonian Height Measurement
danna <-30- UŠ <-60- ninda <-12- kuš3 <-30- šu-si
|Measurement                   | Sexag. (kuš3)   | Reciprocal  |
|------------------------------|-----------------|-------------|
|1/3 kuš3                      | 20              | 3           |
|1/2 kuš3                      | 30              | 2           |
|2/3 kuš3                      | 40              | 1:30        |
|5/6 kuš3                      | 50              | 1:12        |
|(1 diš) kuš3                  | 1               | 1           |
|(1 diš) 1/6 kuš3              | 1:10            | --igi nu--  |
|(1 diš) 1/3 kuš3              | 1:20            | 45          |
|(1 diš) 1/2 kuš3              | 1:30            | 40          |
|(1 diš) 2/3 kuš3              | 1:40            | 36          |
|(1 diš) 5/6 kuš3              | 1:50            | --igi nu--  |
|1/6 ninda                     | 2               | 30          |


The following are the options for the `.metrolist()` method:

In [7]:
help(bh.metrolist)

Help on method metrolist in module mesomath.npvs:

metrolist(*args, **kwargs) class method of __main__.bh
    Generate a list of metrological values for the current class.
    Now supports multi-range sections by passing lists to mmax and step.
    Accepts the same arguments as `metro_generator` and some that are specific to it.



## Late Babylonian Period Metrology


**MesoMath** is designed to work with the metrology of the Old Babylonian period, but it can be extended to use the metrology of other periods. For example, for the Late Babylonian Period, we can start by defining a class `LBcap` for the capacities:

In [8]:
class LBcap(Bcap):  # Capacity
    """This class implement Non-Place-Value System arithmetic
    for Late Babylonian Period capacity units:

        **gur <-5- bariga <-6- ban2 <-10- sila3 <-10- GAR**

    """

    title: str = "Late Babylonian capacity measurement"
    uname: list[str] = "gar sila ban bariga gur".split()
    aname: list[str] = "GAR sila3 ban2 bariga gur".split()
    ufact: list[int] = [10, 10, 6, 5]
    cfact: list[int] = [1, 10, 100, 600, 3000]
    siv: float = 0.1
    siu: str = "litres"
    ubase: int = 3  # bariga

    def vol(self) -> object:
        """Convert capacity to volume measurement

        :return: volume measurement
        :rtype: "Bvol"
        """
        return LBvol(int(round(self.dec/(100/6))))

class LBvol(Bvol):  # Volume
    """This class implement Non-Place-Value System arithmetic
    for Late Babylonian Period volume units:

        **GAN2 <-100- sar <-60- gin2 <-180- še**

    """
    title: str = "Late Babylonian volume measurement"
    
    def cap(self) -> object:
        """Convert volume to capacity measurement"""
        return LBcap(int(round(self.dec*(100/6))))

and then:

In [9]:
a = LBcap('1000 sila')
b = a.vol()
print(f"{b =}")

b =3 gin 60 se


In [10]:
b.explain() 

This is a Late Babylonian volume measurement: 3 gin 60 se
    Metrology:  gan <-100- sar <-60- gin <-180- se
    Factor with unit 'se':  1 180 10800 1080000
Measurement in terms of the smallest unit: 600 (se)
Sexagesimal floating value of the above: 10
Approximate SI value: 0.9999999999999999 cube meters


In [11]:
c = b.cap() 
print(f"{c =}")

c =3 gur 1 bariga 4 ban


In [12]:
c.SI() 

'1000.0 litres'

In [13]:
c.explain() 

This is a Late Babylonian capacity measurement: 3 gur 1 bariga 4 ban
    Metrology:  gur <-5- bariga <-6- ban <-10- sila <-10- gar
    Factor with unit 'gar':  1 10 100 600 3000
Measurement in terms of the smallest unit: 10000 (gar)
Sexagesimal floating value of the above: 2:46:40
Approximate SI value: 1000.0 litres


In [14]:
LBcap.metrolist('1 bariga','3 bariga', '1 ban',1)


Late Babylonian capacity measurement
gur <-5- bariga <-6- ban <-10- sila <-10- gar
|Measurement         |
|--------------------|
|1 bariga            | 1               | 1           |
|1 bariga 1 ban      | 1:10            | --igi nu--  |
|1 bariga 2 ban      | 1:20            | 45          |
|1 bariga 3 ban      | 1:30            | 40          |
|1 bariga 4 ban      | 1:40            | 36          |
|1 bariga 5 ban      | 1:50            | --igi nu--  |
|2 bariga            | 2               | 30          |
|2 bariga 1 ban      | 2:10            | --igi nu--  |
|2 bariga 2 ban      | 2:20            | --igi nu--  |
|2 bariga 3 ban      | 2:30            | 24          |
|2 bariga 4 ban      | 2:40            | 22:30       |
|2 bariga 5 ban      | 2:50            | --igi nu--  |
|3 bariga            | 3               | 20          |


etc. but we should also redefine the rest of the classes to ensure consistency in the operations with the new units.